# Clase: Filtros, Transformación de datos y Feature Engineering

En esta clase vamos a continuar trabajando con el dataset **`satis_clientes.csv`**.

## Objetivos de la clase

- Normalizar y renombrar títulos de columnas
- Transformar tipos de datos
- Realizar selecciones y filtros con Pandas
- Incorporar el método `query()`
- Aplicar transformaciones sobre los datos
- Crear nuevas variables mediante **Feature Engineering**

In [ ]:
import pandas as pd
import numpy as np

## 1. Carga del dataset

Las transformaciones de este Notebook deberían hacerse sobre el dataframe limpio `airbnb_jr_clean.csv` que construimos en la clase pasada, sin duplicados ni nulos.

Pero para entender los problemas de aplicar transformaciones sin limpiar antes los datos, vamos a comenzar usar el dataset con duplicados y nulos.

Se sugiere, una vez entendido todo el Notebook, ajustar el dataset.

In [ ]:
# pd.read_csv recibe la url del dataset
df = pd.read_csv('https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr.csv')
print("Dataset cargado exitosamente! \nDimensiones:", df.shape)

In [ ]:
# Link generado al compartir el archivo
# path = "airbnb_jr_clean.csv"

# Lo importamos usando el método read_csv() con la ruta path
# df = pd.read_csv(path)
# df.shape

## 2. Exploración inicial

Antes de transformar los datos, conviene revisar su estructura general.

In [ ]:
df.info()

## 3. Normalización y renombrado de títulos de columnas

En muchos datasets, los nombres de columnas pueden tener:
- espacios innecesarios
- mayúsculas y minúsculas mezcladas
- tildes o caracteres especiales
- nombres poco prácticos para programar

Por eso, una buena práctica es **normalizar** los títulos para dejarlos prolijos y consistentes.

In [ ]:
df.columns

In [ ]:
# Normalizamos nombres de columnas:
# - quitamos espacios laterales
# - pasamos a minúsculas
# - reemplazamos espacios por guiones bajos
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
)

df.columns

A veces además de normalizar necesitamos **renombrar** columnas para que sean más claras o más cómodas de usar.

In [ ]:
# Renombrado manual de columnas (ajustar según necesidad del dataset)
df = df.rename(columns={
    'neighbourhood': 'barrio'
})

df.columns

## 4. Transformación de tipos de datos

Pandas puede interpretar una columna con un tipo de dato que no sea el más conveniente para analizar.

Por ejemplo:
- una fecha puede venir como texto
- un precio puede venir como string

Corregir los tipos de datos mejora la calidad del análisis y evita errores posteriores.

### 4.1 Convertir dtype `object` a `string`

Recordemos que al momento de importar un dataset, Python asume el tipo de dato. 
En Pandas, una columna con `dtype="object"` no necesariamente contiene únicamente strings. Puede contener strings, números, None u objetos mezclados.

Por eso puede ser útil convertir explícitamente a string cuando necesitás aplicar operaciones de texto de forma consistente.

[astype documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)

In [ ]:
# Ejemplo: asegurar que barrio sea texto
df['barrio'] = df['barrio'].astype('string')
df["barrio"].dtype

### 4.2 Convertir columnas a `int` o `float`

Podríamos usar el mismo método `astype` pero con el argumento apropiado:

In [ ]:
df['bedrooms'] = df['bedrooms'].astype('int')
df['bedrooms'].dtype

Sin embargo, en caso que la cadena de caracteres tenga valores nulos o no sea convertible, generará un error. Para ello se puede usar el atributo `errors`.

En la documentación, el parámetro errors admite:
* 'raise' -> Genera una exception
* 'ignore' -> ignora, no convierte

In [ ]:
# Trabajamos con una copia para explorar las opciones:
df_test = df.copy()
# df_test.dropna(inplace=True)
# print(f"bedrooms nulos {df_test['bedrooms'].isnull().sum()}")
df_test['bedrooms'] = df_test['bedrooms'].astype('int', errors="ignore")
df_test['bedrooms'].dtype

### 4.3 Convertir columnas numéricas con `to_numeric()`

Cuando una variable numérica viene como texto, podemos convertirla con `pd.to_numeric()`.

[to_numeric documentation](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html#pandas.to_numeric)

En la documentación, el parámetro errors admite:
* 'raise' -> Genera una exception
* 'coerce' -> convierte a NaN

Explorar `downcast`

In [ ]:
# Ejemplo: convertir bedrooms a numérico
df_test = df.copy()
# df_test = df_test.dropna()
# print(df_test['bedrooms'].isnull().sum())
df_test['bedrooms'] = pd.to_numeric(df_test['bedrooms'], errors="raise")
print(f"(nulos para bedrooms: {df_test['bedrooms'].isnull().sum()}")

df_test['bedrooms'].dtype

Probar ahora con la variable `price`

In [ ]:
df_test = df.copy()
# df_test = df_test.dropna()
# print(df_test['bedrooms'].isnull().sum())
df_test['price'] = pd.to_numeric(df_test['price'], errors='coerce')
print(f"(nulos para price: {df_test['price'].isnull().sum()}")
df_test['price'].dtype

### 4.4 Convertir fechas con `to_datetime()`

[to_datetime documentation](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html#pandas.to_datetime)

In [ ]:
df['last_review'].head(3)

In [ ]:
# Ejemplo: convertir fecha a formato datetime
df_test = df.copy()
df_test.dropna(inplace=True)
df_test['last_review'] = pd.to_datetime(df_test['last_review'], format="%Y-%m-%d", errors='coerce')
df_test['last_review'].dtype

In [ ]:
# acceder al año directamente
df_test['last_review'].dt.year.head()

In [ ]:
# acceder al mes directamente
df_test['last_review'].dt.month.head(3)

## 5. Selecciones con Pandas

Las **selecciones** permiten elegir columnas o filas específicas del DataFrame.

In [ ]:
# Seleccionar una sola columna
df['barrio']

In [ ]:
# Seleccionar múltiples columnas
df[['barrio', 'bedrooms', 'price']]

In [ ]:
# Seleccionar un conjunto de filas por índice (asociar con slicing en Python)
df.loc[0:5]

In [ ]:
# Seleccionar un subconjunto de filas y columnas
df.loc[0:4, ['barrio', 'bedrooms', 'price']]

## 6. Transformaciones de datos

Las transformaciones modifican el contenido o la estructura del DataFrame para prepararlo mejor para el análisis.

Las transformaciones pueden sobrescribir los datos de la columna o bien podemos generar una nueva.

### 7.1 Transformar valores numéricos

In [ ]:
df.columns

In [ ]:
# Transformar price removiendo el literal $
df_test = df.copy()
df_test['price'] = df_test['price'].str.replace("$", "")

print(f"df['price'].dtype: {df_test['price'].dtype}")
df_test.head(2)

In [ ]:
# Ahora si podemos convertir a integer
df_test['price'] = pd.to_numeric(df_test['price'], errors="coerce", downcast='float')
print(f"df['price'].dtype: {df_test['price'].dtype}")
print(f"nulos en price: {df_test['price'].isnull().sum()}")
df_test.head(2)

Después de eliminar el literal `$` que sucedió al querer convertir a `float`.

Intentá remover el literal `,` usando `.str.replace(",", "")` ahora que sucede?

### 7.3 Transformar cadenas de texto

In [ ]:
# Estandarizar textos usando métodos simples
df_test = df.copy()
df_test['barrio'] = (
        df_test['barrio']
        .str.strip()
        .str.lower()
)

# otros métodos posibles: upper(), title(), capitalize()

df_test.head()

,host_id,latitude,longitude,barrio,room_type,bedrooms,price,number_reviews,last_review,review_scores
0,513802168,-34.602110,-58.406720,Balvanera,Entire home/apt,1.0,"$86,113.00",0,NaN,NaN
1,97542159,-34.601930,-58.377770,San nicolas,Entire home/apt,1.0,"$69,519.86",56,2026-03-02,4.73
2,14450833,-34.582920,-58.423730,Palermo,Entire home/apt,2.0,"$278,114.00",13,2026-07-08,4.92
3,49480723,-34.585860,-58.422580,Palermo,Entire home/apt,2.0,"$190,948.00",12,2026-06-06,4.42
4,50664146,-34.609396,-58.437037,Caballito,Entire home/apt,NaN,"$64,083.00",0,NaN,NaN


In [ ]:
# Estandarizar textos usando estructuras avanzadas

def capitalizar_oraciones(texto):
    oraciones = texto.split(".")
    oraciones = [o.strip().capitalize() for o in oraciones if o.strip()]
    return ". ".join(oraciones)

df_pp["comentarios_pp"] = df_pp["comentarios"].apply(capitalizar_oraciones)

df_pp["comentarios_pp"].head(2)

### 7.4 Ordenar datos

In [ ]:
# Ordenar por calificación de mayor a menor
df_pp.sort_values(by='calificacion', ascending=False) # no sobrescribe nada
# df_pp = df_pp.sort_values(by='calificacion', ascending=True) # sobrescribe
# df_pp.sort_values(by='calificacion', ascending=True, inplace=True) # inplace=True sobrescribe el mismo dataframe
df_pp.head()

In [ ]:
# Ordenar por calificación de mayor a menor y por fecha
df_pp.sort_values(by=['calificacion', 'fecha'], ascending=False, inplace=True)
df_pp.head()

## 8. Feature Engineering

El **Feature Engineering** consiste en crear nuevas variables a partir de columnas existentes para enriquecer el análisis.

Estas nuevas variables pueden ayudar a:
- segmentar mejor los datos
- resumir información
- preparar el dataset para modelos o análisis posteriores

### 8.1 Crear una variable binaria

In [ ]:
# Cliente satisfecho: True si la calificación es alta
df_pp['cliente_satisfecho'] = df_pp['calificacion'] >= 4
df_pp[['calificacion', 'cliente_satisfecho']].sample(5)

### 8.2 Crear una variable usando Apply

In [ ]:
# Cliente satisfecho: True si la calificación es alta, alternativa usando Apply
df_pp['cliente_estado'] = df_pp['calificacion'].apply(
    lambda x: 'Satisfecho' if x >= 4 else 'Insatisfecho'
)
df_pp[['calificacion', 'cliente_estado']].sample(5)

### 8.2 Crear una categoría a partir de una variable numérica

In [ ]:
def clasificar_calificacion(x):
    if pd.isna(x):
        return 'Sin dato'
    elif x < 3:
        return 'Baja'
    elif x < 4:
        return 'Media'
    else:
        return 'Alta'

if 'calificacion' in df_pp.columns:
    df_pp['nivel_satisfaccion'] = df_pp['calificacion'].apply(clasificar_calificacion)

df_pp[['calificacion', 'nivel_satisfaccion']].head()

### 8.3 Crear una variable temporal a partir de una fecha

In [ ]:
# Extraer componentes de fecha
if 'fecha' in df_pp.columns:
    df_pp['anio'] = df_pp['fecha'].dt.year
    df_pp['mes'] = df_pp['fecha'].dt.month

columnas_mostrar = [col for col in ['fecha', 'anio', 'mes'] if col in df_pp.columns]
df_pp[columnas_mostrar].head()

### 8.4 Crear una variable a partir del texto

In [ ]:
# Largo del comentario como nueva variable
df_pp['largo_comentario'] = df_pp['comentarios'].fillna('').str.len()

columnas_mostrar = [col for col in ['comentarios', 'largo_comentario'] if col in df_pp.columns]
df_pp[columnas_mostrar].head()

### 9. Filtros

In [ ]:
df_pp.columns

In [ ]:
# Filtrar registros con calificación mayor a 4
valor_corte = 2
df_pp[df_pp["calificacion"] > valor_corte] # select * from df_pp where calificacion > 4
df_pp[df_pp["calificacion"] > valor_corte][["calificacion", "comentarios"]].head(2) # select calificacion, comentarios from df_pp where calificacion > 4

In [ ]:
df_pp["comentarios"]

In [ ]:
# Filtro con múltiples condiciones
# Filtrar los registros con calificacion mayor a 4
# & "sin comentarios" en la columna comentarios
valor_corte = 4
# df_pp[(df_pp["calificacion"] > valor_corte) & (df_pp["comentarios"].str.lower()== "sin comentario") ].head(2)
df_pp[(df_pp["calificacion"] > valor_corte) & (df_pp["comentarios_pp"].str.contains("sin", case=False)) ].head(2)

In [ ]:
# Filtro con múltiples condiciones
# Filtrar los registros con calificacion mayor a 4
# y que la columna empresa contenga LLC
df_pp[
    (df_pp['calificacion'] > 4) &
    (df_pp['empresa'].str.contains("llc", case=False))
].head()

In [ ]:
# Veamos el rango de fechas de los datos
df_pp.describe(include=['datetime'])

In [ ]:
df_pp["fecha"].dtypes

In [ ]:
df_pp[df_pp["fecha"] > "2024-06-01"].sort_values(by="fecha", ascending=True).head(2)

In [ ]:
# Apliquemos algunos filtros basados en fecha
fecha_corte = pd.to_datetime('2024-06-01')
print(fecha_corte)
# df_pp['fecha'] > fecha_corte
# df_pp[df_pp['fecha'] > fecha_corte]
df_pp[df_pp['fecha'] > fecha_corte].sort_values(by='fecha', ascending=True).head(3)

In [ ]:
# Filtramos solo las ventas de un año
# df_pp[df_pp['fecha'].dt.year == 2023].sort_values(by='fecha', ascending=True).head()
df_pp[(df_pp['fecha'].dt.year == 2024) & (df_pp['fecha'].dt.month == 12)].sort_values(by='fecha', ascending=True).head()

In [ ]:
# Filtramos solo las ventas de un año y un mes
df_pp[(df_pp['fecha'].dt.year == 2024) & (df_pp['fecha'].dt.month == 1)].sort_values(by='fecha', ascending=True).head()

## 10. Cierre

En esta clase trabajamos sobre un flujo muy habitual en análisis de datos:

1. revisar el dataset
2. normalizar nombres de columnas
3. corregir tipos de datos
4. transformar el DataFrame
5. crear nuevas variables de análisis

Estos pasos son fundamentales para preparar datos antes de hacer análisis más avanzados, visualizaciones o modelos.